# 第 5 章 · Tool Use（Google Search / ADK）

`google_search` 是**内建工具**：搜索在 Google 侧执行，`tools=[]` 里放的是框架导出的工具对象，
不是你写的函数。

所以「模型到底调没调工具」代码里看不出来——只能查 `grounding_metadata`。
这也正是本章那条规律：**工具越内建，可观测性越差**。

kernel `Python (agentic-adk)`。


In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import quiet_adk_sync_warnings
quiet_adk_sync_warnings()          # 【适配】消掉 ADK 同步 API 的弃用提示
from common import adk_model, short_error, retry_call
from google.adk.agents import LlmAgent          # 【适配】书里 import Agent 却用 ADKAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

USER_ID = "user1234"

# ★ 本章新增：内建工具。不是本地函数，搜索发生在 Google 侧。
root_agent = LlmAgent(
    name="basic_search_agent",
    model=adk_model(),                          # 【适配】gemini-2.0-flash-exp 已下线
    description="Agent to answer questions using Google Search.",
    instruction="I can answer your questions by searching the internet. Just ask me anything!",
    tools=[google_search],
)

runner = InMemoryRunner(root_agent)
session_id = str(uuid.uuid4())
# 【适配】create_session 是协程 → _sync 版；书里那句 asyncio.run 在 notebook 里会冲突
runner.session_service.create_session_sync(
    app_name=runner.app_name, user_id=USER_ID, session_id=session_id
)


def text_of(content) -> str:                    # 【适配】Content 没有 .text 属性
    if getattr(content, "text", None):
        return content.text
    return "".join(p.text for p in (getattr(content, "parts", None) or [])
                   if getattr(p, "text", None))


def collect():
    return list(runner.run(
        user_id=USER_ID,
        session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text="what's the latest ai news?")]),
    ))


try:
    events = retry_call(collect, label="搜索 ")
    answered = 0
    for event in events:
        if event.is_final_response() and event.content:
            answered += 1
            print("Agent Response:", text_of(event.content).strip())

            # 内建工具调没调，代码里看不到，只能靠 grounding 判断
            gm = getattr(event, "grounding_metadata", None)
            print("[grounding] 搜索词:", list(getattr(gm, "web_search_queries", None) or []) if gm else "(无)")
            print("[grounding] 引用来源:", len(getattr(gm, "grounding_chunks", None) or []) if gm else 0)
    if answered == 0:
        # 中转站偶发返回空回合；不提示的话这里会一声不响 —— 别让它静默
        print(f"⚠️ 收到 {len(events)} 个事件，但没有 final response。重跑这一格即可。")
except Exception as e:
    print(f"❌ {type(e).__name__}: {short_error(e)}")


Agent Response: The landscape of artificial intelligence is moving faster than ever, with major developers shifting their focus from simple chatbots to autonomous AI agents, massive cost reductions, and high-stakes financial milestones. 

The most prominent updates in the AI sector include:

---

### 1. OpenAI: GPT-6.1 Terra, Rosalind Research, and "Compute Campuses"
* **New Models:** OpenAI recently expanded its GPT-6 lineup:
  * **GPT-6.1 Terra** (released October 26, 2026) is a highly efficient mid-tier model designed to compete directly with Anthropic’s Claude Sonnet. Optimized for visual reasoning, computer use (interacting with screens and apps), and business automation, it costs $1 per million input tokens—roughly a tenth of the price of OpenAI’s top-tier Astra model.
  * **GPT-6 Rosalind Research** (released October 30, 2026) is a restricted-access model optimized for complex, multi-step life-sciences and academic research.
* **Rebranding Data Centers:** In a bid to shift publi